# JOSS Paper Figures

This notebook generates all four figures for the HyPlan JOSS paper.

| Figure | Description | Data sources |
|--------|-------------|--------------|
| Fig 1 | Workflow overview diagram | None (schematic) |
| Fig 2 | Terrain-aware flight box — Rincón de la Vieja | Copernicus GLO-30 DEM |
| Fig 3 | Complete mission — still air vs wind | Airport database |
| Fig 4 | Cloud climatology + vegetation phenology | Open-Meteo ERA5, MODIS NDVI |

**Outputs:** Each figure is saved as both PNG (300 dpi) and PDF in this directory.

**Credentials:** Figure 4 requires NASA EarthData credentials for NDVI (falls back to cached data if unavailable).

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import os
import tempfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import LightSource
import geopandas as gpd
import pandas as pd
import rasterio
from shapely.geometry import box, shape

from hyplan import (
    AVIRIS3, KingAirB200,
    Airport, initialize_data,
    box_around_polygon_terrain,
    compute_flight_plan, ConstantWindField,
    generate_swath_polygon, calculate_swath_widths,
    ureg,
)
from hyplan.geometry import minimum_rotated_rectangle, rectangle_dimensions
from hyplan.terrain import generate_demfile

# Output directory (same directory as this notebook)
FIG_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" not in dir() else os.path.dirname(__file__)
FIG_DIR = os.path.abspath(".")

%matplotlib inline

## Figure 1: Workflow Overview Diagram

A pipeline diagram showing the five-stage workflow with side inputs
grouped into "How to fly" and "When to fly" categories.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-2.8, 3.8)
ax.set_aspect("equal")
ax.axis("off")

# Colors
C_STAGE = "#2c3e50"
C_STAGE_TXT = "white"
C_HOW = "#e67e22"
C_WHEN = "#2980b9"
C_OUTPUT = "#27ae60"
C_ARROW = "#7f8c8d"
C_BG_HOW = "#fef3e2"
C_BG_WHEN = "#e8f4fd"


def stage_box(x, y, label, color=C_STAGE, text_color=C_STAGE_TXT, w=1.7, h=0.7):
    box = FancyBboxPatch(
        (x - w / 2, y - h / 2), w, h,
        boxstyle="round,pad=0.08", facecolor=color,
        edgecolor="none", zorder=3,
    )
    ax.add_patch(box)
    ax.text(x, y, label, ha="center", va="center", fontsize=7.5,
            fontweight="bold", color=text_color, zorder=4)


def side_label(x, y, label, color, fontsize=6.5):
    ax.text(x, y, label, ha="center", va="center", fontsize=fontsize,
            color=color, fontstyle="italic", zorder=4)


def arrow(x1, y1, x2, y2, color=C_ARROW):
    ax.annotate(
        "", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.2),
        zorder=2,
    )


def side_arrow(x1, y1, x2, y2, color):
    ax.annotate(
        "", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="-|>", color=color, lw=0.8,
                        linestyle="--"),
        zorder=2,
    )


def iter_arrow(x1, y1, x2, y2, color, rad=0.3):
    ax.annotate(
        "", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="<|-|>", color=color, lw=0.9,
                        linestyle="--",
                        connectionstyle=f"arc3,rad={rad}"),
        zorder=2,
    )


# Main pipeline
xs = [0.8, 3.0, 5.2, 7.4, 9.6]
labels = ["Study\nArea", "Flight\nLines", "Swath\nAnalysis", "Mission\nPlan", "Outputs"]
colors = [C_STAGE, C_STAGE, C_STAGE, C_STAGE, C_OUTPUT]
y_main = 1.0

for i, (x, lab, c) in enumerate(zip(xs, labels, colors)):
    stage_box(x, y_main, lab, color=c)
    if i < len(xs) - 1:
        arrow(x + 0.9, y_main, xs[i + 1] - 0.9, y_main)

# Output sub-labels
outputs = ["KML / GPX", "ForeFlight CSV", "Excel / ICARTT", "Folium maps"]
for i, txt in enumerate(outputs):
    ax.text(9.6, y_main - 0.55 - i * 0.32, txt, ha="center", va="top",
            fontsize=5.5, color="#1a7a3a", zorder=4)

# "How to fly" inputs (orange, top)
how_y = 2.8
bg_how = FancyBboxPatch(
    (1.5, how_y - 0.45), 7.4, 0.9,
    boxstyle="round,pad=0.1", facecolor=C_BG_HOW,
    edgecolor=C_HOW, linewidth=0.8, linestyle="--", zorder=1,
)
ax.add_patch(bg_how)
ax.text(1.7, how_y + 0.35, "How to fly", fontsize=7, fontweight="bold",
        color=C_HOW, va="center", zorder=4)

how_positions = [(3.0, how_y), (5.2, how_y + 0.15), (6.8, how_y + 0.15), (8.0, how_y - 0.15)]
how_labels = ["Sensor", "Terrain / DEM", "Aircraft", "Winds"]
for (hx, hy), lab in zip(how_positions, how_labels):
    side_label(hx, hy, lab, C_HOW)
    target_x = min(xs, key=lambda sx: abs(sx - hx))
    side_arrow(hx, hy - 0.35, target_x, y_main + 0.4, C_HOW)

# "When to fly" inputs (blue, bottom)
when_y = -0.8
bg_when = FancyBboxPatch(
    (-0.3, when_y - 0.45), 9.2, 0.9,
    boxstyle="round,pad=0.1", facecolor=C_BG_WHEN,
    edgecolor=C_WHEN, linewidth=0.8, linestyle="--", zorder=1,
)
ax.add_patch(bg_when)
ax.text(-0.05, when_y + 0.35, "When to fly", fontsize=7, fontweight="bold",
        color=C_WHEN, va="center", zorder=4)

side_label(1.1, when_y + 0.15, "Vegetation\nphenology", C_WHEN)
side_arrow(1.1, when_y + 0.5, xs[0] + 0.15, y_main - 0.4, C_WHEN)

side_label(2.6, when_y + 0.15, "Cloud\nclimatology", C_WHEN)
side_arrow(2.6, when_y + 0.5, xs[0] + 0.5, y_main - 0.4, C_WHEN)
iter_arrow(2.9, when_y + 0.5, xs[3] - 0.3, y_main - 0.4, C_WHEN, rad=-0.25)

side_label(5.8, when_y + 0.15, "Solar\ngeometry", C_WHEN)
side_arrow(5.8, when_y + 0.5, xs[3] - 0.4, y_main - 0.4, C_WHEN)

side_label(7.6, when_y + 0.15, "Cloud\nforecasts", C_WHEN)
side_arrow(7.6, when_y + 0.5, xs[3] + 0.1, y_main - 0.4, C_WHEN)

fig.tight_layout(pad=0.5)
fig.savefig(os.path.join(FIG_DIR, "fig1_workflow.png"), dpi=300,
            bbox_inches="tight", facecolor="white")
fig.savefig(os.path.join(FIG_DIR, "fig1_workflow.pdf"),
            bbox_inches="tight", facecolor="white")
plt.show()
print("Figure 1 saved.")

## Figure 2: Terrain-Aware Flight Box — Rincón de la Vieja

Four-panel figure showing the spatial planning pipeline over a volcanic
national park with irregular boundaries and significant terrain relief:

- **(a)** Study area polygon with terrain
- **(b)** Minimum rotated bounding rectangle + sensor parameters
- **(c)** Flight lines with terrain-aware spacing
- **(d)** Clipped lines with terrain-aware swath polygons

In [ ]:
sensor = AVIRIS3()
flight_altitude = ureg.Quantity(25000, "feet")

# Rincón de la Vieja National Park, Costa Rica
gdf = gpd.read_file(os.path.join(FIG_DIR, "rincon_de_la_vieja.geojson"))
study_area = gdf.geometry.iloc[0]
centroid = study_area.centroid

# Rotated rectangle
min_rect = minimum_rotated_rectangle(study_area)
lat0, lon0, azimuth, length_m, width_m = rectangle_dimensions(min_rect, azimuth=None)

# Swath width (flat-earth reference)
swath_w = sensor.swath_width(flight_altitude)

# DEM
bounds = study_area.bounds
dem_file = generate_demfile(
    np.array([bounds[1], bounds[3]]),
    np.array([bounds[0], bounds[2]]),
)

# Terrain-aware flight lines (unclipped)
lines_unclipped = box_around_polygon_terrain(
    instrument=sensor,
    altitude_msl=flight_altitude,
    polygon=study_area,
    box_name="RDV",
    overlap=20.0,
    alternate_direction=True,
    clip_to_polygon=False,
)

# Clipped to buffered polygon
clip_buffer_deg = 0.012
buffered_area = study_area.buffer(clip_buffer_deg)

lines_clipped = box_around_polygon_terrain(
    instrument=sensor,
    altitude_msl=flight_altitude,
    polygon=study_area,
    box_name="RDV",
    overlap=20.0,
    alternate_direction=True,
    clip_to_polygon=True,
    clip_polygon=buffered_area,
)

# Terrain-aware swaths
swaths = []
for fl in lines_clipped:
    try:
        sw = generate_swath_polygon(fl, sensor, along_precision=500.0, dem_file=dem_file)
        swaths.append(sw)
    except Exception:
        swaths.append(None)

# DEM shaded relief
with rasterio.open(dem_file) as src:
    dem_data = src.read(1)
    dem_bounds = src.bounds
    valid = dem_data[dem_data > 0]
    elev_min, elev_max = int(valid.min()), int(valid.max())

ls = LightSource(azdeg=315, altdeg=35)
rgb = ls.shade(dem_data, cmap=plt.cm.terrain, vert_exag=2, blend_mode="soft")
fade = 0.4
rgb[..., :3] = rgb[..., :3] * (1 - fade) + fade
dem_extent = [dem_bounds.left, dem_bounds.right, dem_bounds.bottom, dem_bounds.top]

# Figure
fig, axes = plt.subplots(2, 2, figsize=(7, 7))
(ax_a, ax_b, ax_c, ax_d) = axes.flat

pad = 0.015
xlim = (bounds[0] - pad, bounds[2] + pad)
ylim = (bounds[1] - pad, bounds[3] + pad)

for ax in axes.flat:
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal")
    ax.tick_params(labelsize=5.5)

ann_bbox = dict(boxstyle="round,pad=0.12", facecolor="white",
                edgecolor="none", alpha=0.85)

sx, sy = study_area.exterior.xy

# Panel (a): Study area with terrain
ax_a.imshow(rgb, extent=dem_extent, origin="upper", aspect="auto", zorder=0)
ax_a.plot(sx, sy, color="black", linewidth=1.5, zorder=3)
ax_a.set_title("(a) Study area", fontsize=8, fontweight="bold")
ax_a.set_ylabel("Latitude", fontsize=7)
ax_a.text(centroid.x, bounds[1] + 0.006, "Rincón de la Vieja\nNational Park",
          ha="center", va="bottom", fontsize=5.5, fontstyle="italic",
          bbox=ann_bbox, zorder=4)
ax_a.text(0.97, 0.03, f"Elev: {elev_min}\u2013{elev_max} m",
          transform=ax_a.transAxes, fontsize=5.5, ha="right", va="bottom",
          bbox=ann_bbox, zorder=5)

# Panel (b): Bounding rectangle + sensor info
ax_b.imshow(rgb, extent=dem_extent, origin="upper", aspect="auto", zorder=0)
ax_b.plot(sx, sy, color="black", linewidth=1, zorder=2)
rx, ry = min_rect.exterior.xy
ax_b.plot(rx, ry, color="#e74c3c", linewidth=1.5, linestyle="-", zorder=3)
ax_b.set_title("(b) Bounding rectangle + sensor", fontsize=8, fontweight="bold")
ax_b.text(0.03, 0.97,
          f"AVIRIS-3 @ {flight_altitude.to(ureg.feet):.0f}\n"
          f"Swath: {swath_w.to(ureg.km):.1f}\n"
          f"Overlap: 20%\n"
          f"Azimuth: {azimuth:.0f}\u00b0\n"
          f"Lines: {len(lines_unclipped)} (terrain-aware)",
          transform=ax_b.transAxes, fontsize=5.5, va="top",
          bbox=ann_bbox, zorder=5)

# Panel (c): Unclipped flight lines
ax_c.imshow(rgb, extent=dem_extent, origin="upper", aspect="auto", zorder=0)
ax_c.plot(sx, sy, color="black", linewidth=1, zorder=2)

for fl in lines_unclipped:
    x = [fl.waypoint1.longitude, fl.waypoint2.longitude]
    y = [fl.waypoint1.latitude, fl.waypoint2.latitude]
    ax_c.plot(x, y, color="white", linewidth=1.5, zorder=3)
    ax_c.plot(x, y, color="#2c3e50", linewidth=0.8, zorder=4)
    mx, my = np.mean(x), np.mean(y)
    ax_c.annotate("", xy=(x[1], y[1]), xytext=(mx, my),
                  arrowprops=dict(arrowstyle="-|>", color="#2c3e50", lw=0.8),
                  zorder=5)

ax_c.set_title("(c) Terrain-aware line spacing", fontsize=8, fontweight="bold")
ax_c.set_ylabel("Latitude", fontsize=7)
ax_c.set_xlabel("Longitude", fontsize=7)

# Panel (d): Clipped lines + terrain-aware swaths
ax_d.imshow(rgb, extent=dem_extent, origin="upper", aspect="auto", zorder=0)
ax_d.plot(sx, sy, color="black", linewidth=1, linestyle="-", alpha=0.8, zorder=2)

for sw in swaths:
    if sw is not None and sw.is_valid:
        swx, swy = sw.exterior.xy
        ax_d.fill(swx, swy, alpha=0.45, color="#3498db", edgecolor="#2980b9",
                  linewidth=0.5, zorder=3)

for fl in lines_clipped:
    x = [fl.waypoint1.longitude, fl.waypoint2.longitude]
    y = [fl.waypoint1.latitude, fl.waypoint2.latitude]
    ax_d.plot(x, y, color="white", linewidth=1.2, zorder=4)
    ax_d.plot(x, y, color="#2c3e50", linewidth=0.6, zorder=5)

ax_d.set_title("(d) Clipped + terrain swaths", fontsize=8, fontweight="bold")
ax_d.set_xlabel("Longitude", fontsize=7)

all_widths = [calculate_swath_widths(sw) for sw in swaths if sw is not None and sw.is_valid]
if all_widths:
    w_min = min(w["min_width"] for w in all_widths)
    w_max = max(w["max_width"] for w in all_widths)
    ax_d.text(0.03, 0.03,
              f"Swath width: {w_min:.0f}\u2013{w_max:.0f} m",
              transform=ax_d.transAxes, fontsize=5.5,
              bbox=ann_bbox, zorder=6)

fig.suptitle("Terrain-Aware Flight Box \u2014 Rinc\u00f3n de la Vieja, Costa Rica",
             fontsize=10, fontweight="bold", y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(os.path.join(FIG_DIR, "fig2_flight_box.png"), dpi=300,
            bbox_inches="tight", facecolor="white")
fig.savefig(os.path.join(FIG_DIR, "fig2_flight_box.pdf"),
            bbox_inches="tight", facecolor="white")
plt.show()
print("Figure 2 saved.")

## Figure 3: Complete Mission — Still Air vs Wind

Uses `compute_flight_plan` over Rincón de la Vieja with MRLB (Liberia)
as the departure/return airport. Compares still-air Dubins transit arcs
against trochoidal arcs in 60 kt northeasterly wind.

- **Top row:** Map views (still air vs wind)
- **Bottom:** Altitude profile overlay

In [ ]:
sensor = AVIRIS3()
aircraft = KingAirB200()
flight_altitude = ureg.Quantity(25000, "feet")
cruise_speed = aircraft.cruise_speed_at(flight_altitude)
bank_angle = 25.0

gdf = gpd.read_file(os.path.join(FIG_DIR, "rincon_de_la_vieja.geojson"))
study_area = gdf.geometry.iloc[0]

initialize_data(countries=["CR"])
airport = Airport("MRLB")

# Terrain-aware clipped lines (same parameters as Figure 2)
clip_buffer_deg = 0.012
buffered_area = study_area.buffer(clip_buffer_deg)

lines = box_around_polygon_terrain(
    instrument=sensor,
    altitude_msl=flight_altitude,
    polygon=study_area,
    box_name="RDV",
    overlap=20.0,
    alternate_direction=True,
    clip_to_polygon=True,
    clip_polygon=buffered_area,
)

# Wind: 60 kt northeasterly (from NE)
wind_speed = 60 * ureg.knot
wind_field = ConstantWindField(wind_speed=wind_speed, wind_from_deg=45.0)

# Flight plans
plan_still = compute_flight_plan(
    aircraft=aircraft, flight_sequence=lines,
    takeoff_airport=airport, return_airport=airport,
)
plan_wind = compute_flight_plan(
    aircraft=aircraft, flight_sequence=lines,
    takeoff_airport=airport, return_airport=airport,
    wind_source=wind_field,
)

# Segment colors
seg_colors = {
    "takeoff": "#8e44ad", "climb": "#8e44ad",
    "transit": "#e67e22",
    "flight_line": "#2c3e50",
    "descent": "#8e44ad", "approach": "#8e44ad",
}
seg_lw = {
    "takeoff": 1.2, "climb": 1.2,
    "transit": 1.8,
    "flight_line": 2.5,
    "descent": 1.2, "approach": 1.2,
}

# Figure
fig = plt.figure(figsize=(10, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[1.2, 0.8], hspace=0.28, wspace=0.08)
ax_map1 = fig.add_subplot(gs[0, 0])
ax_map2 = fig.add_subplot(gs[0, 1], sharey=ax_map1)
ax_alt = fig.add_subplot(gs[1, :])

# Top row: Map views
for ax, plan, title in [
    (ax_map1, plan_still, "Still air"),
    (ax_map2, plan_wind, "60 kt northeasterly wind"),
]:
    sx, sy = study_area.exterior.xy
    ax.plot(sx, sy, color="gray", linewidth=0.8, linestyle=":", alpha=0.4, zorder=1)
    ax.fill(sx, sy, alpha=0.05, color="green", zorder=0)

    ax.plot(airport.longitude, airport.latitude, "s", color="#8e44ad",
            markersize=6, zorder=6)
    ax.text(airport.longitude - 0.02, airport.latitude + 0.008,
            airport.icao_code, fontsize=8, color="#8e44ad",
            fontweight="bold", zorder=6, ha="right")

    for _, seg in plan.iterrows():
        geom = seg.geometry
        if geom is None or geom.is_empty:
            continue
        seg_type = seg["segment_type"]
        color = seg_colors.get(seg_type, "#bdc3c7")
        lw = seg_lw.get(seg_type, 1.0)
        xs, ys = geom.xy
        ax.plot(xs, ys, color=color, linewidth=lw,
                zorder=3 if seg_type == "flight_line" else 2)

        if seg_type == "flight_line":
            mid = len(xs) // 2
            if mid > 0:
                ax.annotate("", xy=(xs[mid], ys[mid]),
                            xytext=(xs[mid - 1], ys[mid - 1]),
                            arrowprops=dict(arrowstyle="-|>", color=color, lw=1.5),
                            zorder=5)

    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Longitude", fontsize=9)
    ax.tick_params(labelsize=7)

    # Wind arrows (right panel only)
    if plan is plan_wind:
        xlims = ax.get_xlim()
        ylims = ax.get_ylim()
        for wlon in np.arange(xlims[0], xlims[1], 0.05):
            for wlat in np.arange(ylims[0], ylims[1], 0.015):
                ax.annotate("", xy=(wlon - 0.011, wlat - 0.011),
                            xytext=(wlon, wlat),
                            arrowprops=dict(arrowstyle="-|>", color="#e74c3c",
                                            lw=0.5, alpha=0.2), zorder=1)

ax_map1.set_ylabel("Latitude", fontsize=9)
plt.setp(ax_map2.get_yticklabels(), visible=False)

legend_elements = [
    plt.Line2D([0], [0], color="#2c3e50", linewidth=2.5, label="Data collection"),
    plt.Line2D([0], [0], color="#e67e22", linewidth=1.8, label="Transit (Dubins)"),
    plt.Line2D([0], [0], color="#8e44ad", linewidth=1.2, label="Takeoff / landing"),
    plt.Line2D([0], [0], marker="s", color="w", markerfacecolor="#8e44ad",
               markersize=5, label=f"Airport ({airport.icao_code})"),
]
ax_map1.legend(handles=legend_elements, loc="upper left", fontsize=7, framealpha=0.9)
ax_map2.legend(handles=legend_elements, loc="upper left", fontsize=7, framealpha=0.9)

# Bottom: Altitude profile
alt_offset = 500
for plan, label, ls, alpha, offset in [
    (plan_still, "Still air", "-", 0.8, alt_offset),
    (plan_wind, "With wind", "--", 0.5, 0),
]:
    cum_time = 0.0
    for _, seg in plan.iterrows():
        t = seg["time_to_segment"]
        seg_type = seg["segment_type"]
        color = seg_colors.get(seg_type, "#bdc3c7")
        h_start = seg["start_altitude"] + offset
        h_end = seg["end_altitude"] + offset
        ax_alt.plot([cum_time, cum_time + t], [h_start, h_end],
                    color=color, linewidth=1.5 if ls == "-" else 1.0,
                    linestyle=ls, alpha=alpha)
        cum_time += t

alt_legend = [
    plt.Line2D([0], [0], color="#8e44ad", lw=1.5, label="Takeoff / landing"),
    plt.Line2D([0], [0], color="#e67e22", lw=1.5, label="Transit"),
    plt.Line2D([0], [0], color="#2c3e50", lw=1.5, label="Data collection"),
    plt.Line2D([0], [0], color="black", lw=1.5, ls="-", label=f"Still air (+{alt_offset} ft offset)"),
    plt.Line2D([0], [0], color="black", lw=1.0, ls="--", label="With wind"),
]
ax_alt.legend(handles=alt_legend, loc="upper right", fontsize=7, framealpha=0.9)
ax_alt.set_xlabel("Mission time (minutes)", fontsize=9)
ax_alt.set_ylabel("Altitude (ft MSL)", fontsize=9)
ax_alt.set_title("Altitude profile", fontsize=11, fontweight="bold")
ax_alt.set_ylim(-500, flight_altitude.m_as("feet") * 1.15)
ax_alt.tick_params(labelsize=7)
ax_alt.axhline(flight_altitude.m_as("feet"), color="#bdc3c7", linewidth=0.5,
               linestyle=":", zorder=0)

t_still = plan_still["time_to_segment"].sum()
t_wind = plan_wind["time_to_segment"].sum()
ax_alt.text(0.98, 0.15,
            f"Still air: {t_still:.0f} min\nWith wind: {t_wind:.0f} min "
            f"({t_wind - t_still:+.0f})",
            transform=ax_alt.transAxes, fontsize=8, ha="right", va="bottom",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor="#bdc3c7", linewidth=0.5))

fig.text(0.5, 0.99,
         f"King Air B200, {cruise_speed.to(ureg.knot):.0f} cruise, "
         f"{bank_angle}\u00b0 bank"
         f" \u2014 {airport.icao_code} \u2192 Rinc\u00f3n de la Vieja \u2192 {airport.icao_code}",
         ha="center", fontsize=8, color="#555555", va="top")

fig.savefig(os.path.join(FIG_DIR, "fig3_wind_dubins.png"), dpi=300,
            bbox_inches="tight", facecolor="white")
fig.savefig(os.path.join(FIG_DIR, "fig3_wind_dubins.pdf"),
            bbox_inches="tight", facecolor="white")
plt.show()
print("Figure 3 saved.")

## Figure 4: Cloud Climatology + Vegetation Phenology

"When to fly" — combines cloud fraction from Open-Meteo ERA5 (no auth
required) with NDVI from MODIS MOD13A1 via NASA AppEEARS (EarthData
credentials required, falls back to cached CSV if unavailable).

Two Costa Rica sites are compared: the volcanic highlands (Rincón de la
Vieja) and the Guanacaste lowlands. The dry season window (Dec–Apr) is
highlighted as the optimal flight period.

In [ ]:
# Load credentials from .env if available
env_path = os.path.join(FIG_DIR, "..", "..", ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, _, val = line.partition("=")
                os.environ.setdefault(key.strip(), val.strip())

# Month ticks
month_starts = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
month_labels = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]

# Study sites (Costa Rica)
sites = gpd.GeoDataFrame([
    {"Name": "Rincon de la Vieja", "geometry": box(-85.40, 10.72, -85.25, 10.87)},
    {"Name": "Guanacaste lowlands", "geometry": box(-85.70, 10.40, -85.40, 10.60)},
], crs="EPSG:4326")

tmp_poly = os.path.join(tempfile.gettempdir(), "fig4_sites.geojson")
sites.to_file(tmp_poly, driver="GeoJSON")

# Fetch cloud fraction
print("Fetching cloud fraction from Open-Meteo ERA5...")
from hyplan.clouds import fetch_cloud_fraction, summarize_cloud_fraction_by_doy

cloud_df = fetch_cloud_fraction(
    polygon_file=tmp_poly,
    year_start=2010,
    year_stop=2022,
    day_start=1,
    day_stop=365,
    source="openmeteo",
)
cloud_summary = summarize_cloud_fraction_by_doy(cloud_df, window=7)
print(f"  Cloud: {len(cloud_df)} rows, {cloud_df['polygon_id'].nunique()} sites")

# Fetch NDVI (use cache if available)
ndvi_cache = os.path.join(FIG_DIR, "fig4_ndvi_cache.csv")

if os.path.exists(ndvi_cache):
    print(f"Loading cached NDVI from {ndvi_cache}")
    ndvi_df = pd.read_csv(ndvi_cache, parse_dates=["date"])
else:
    print("Fetching NDVI from MODIS via AppEEARS...")
    from hyplan.phenology import fetch_phenology

    ndvi_df = fetch_phenology(
        polygon_file=tmp_poly,
        product="ndvi",
        year_start=2010,
        year_stop=2022,
        source="appeears",
    )
    ndvi_df.to_csv(ndvi_cache, index=False)
    print(f"  Cached to {ndvi_cache}")

from hyplan.phenology import summarize_phenology_by_doy
ndvi_summary = summarize_phenology_by_doy(ndvi_df, window=3)
print(f"  NDVI: {len(ndvi_df)} rows, {ndvi_df['polygon_id'].nunique()} sites")

In [ ]:
fig, (ax_cloud, ax_ndvi) = plt.subplots(
    2, 1, figsize=(6, 4.5), sharex=True,
    gridspec_kw={"hspace": 0.08},
)

site_colors = {
    "Rincon de la Vieja": "#2ecc71",
    "Guanacaste lowlands": "#e67e22",
}

# Dry season window (Dec-Apr)
for ax in (ax_cloud, ax_ndvi):
    ax.axvspan(335, 365, alpha=0.10, color="#e67e22", zorder=0)
    ax.axvspan(1, 105, alpha=0.10, color="#e67e22", zorder=0)

# Top: Cloud fraction
for name, grp in cloud_summary.groupby("polygon_id"):
    grp = grp.sort_values("day_of_year")
    color = site_colors.get(name, "gray")
    ax_cloud.plot(grp["day_of_year"], grp["cloud_fraction_mean"],
                  color=color, linewidth=1.5, label=name)
    if "cloud_fraction_std" in grp.columns:
        ax_cloud.fill_between(
            grp["day_of_year"],
            grp["cloud_fraction_mean"] - grp["cloud_fraction_std"],
            grp["cloud_fraction_mean"] + grp["cloud_fraction_std"],
            alpha=0.12, color=color,
        )

ax_cloud.set_ylabel("Cloud Fraction", fontsize=8)
ax_cloud.set_ylim(0, 1.0)
ax_cloud.legend(fontsize=6.5, loc="lower left", framealpha=0.9)
ax_cloud.set_title("Cloud Climatology and Vegetation Phenology \u2014 Costa Rica",
                    fontsize=10, fontweight="bold", pad=8)
ax_cloud.tick_params(labelsize=7)
ax_cloud.annotate("Dry season", xy=(60, 0.12), fontsize=6, ha="center",
                   color="#c0392b", fontstyle="italic")

# Bottom: NDVI
for name, grp in ndvi_summary.groupby("polygon_id"):
    grp = grp.sort_values("day_of_year")
    color = site_colors.get(name, "gray")
    ax_ndvi.plot(grp["day_of_year"], grp["value_mean"],
                 color=color, linewidth=1.5, label=name)
    if "value_std" in grp.columns:
        ax_ndvi.fill_between(
            grp["day_of_year"],
            grp["value_mean"] - grp["value_std"],
            grp["value_mean"] + grp["value_std"],
            alpha=0.12, color=color,
        )

ax_ndvi.set_ylabel("NDVI", fontsize=8)
ax_ndvi.set_ylim(0, 1.0)
ax_ndvi.set_xlabel("Month", fontsize=8)
ax_ndvi.set_xlim(1, 365)
ax_ndvi.set_xticks(month_starts)
ax_ndvi.set_xticklabels(month_labels)
ax_ndvi.tick_params(labelsize=7)

fig.text(0.15, 0.52, "Dry season\n(optimal)",
         fontsize=7, ha="center", va="center",
         color="#c0392b", fontweight="bold",
         bbox=dict(boxstyle="round,pad=0.3", facecolor="#fdebd0",
                   edgecolor="#c0392b", linewidth=0.8))

fig.text(0.5, 0.01,
         "ERA5 cloud fraction (Open-Meteo) and MODIS NDVI (AppEEARS), 2010\u20132022",
         ha="center", fontsize=6, color="#999999")

fig.subplots_adjust(left=0.1, right=0.95, bottom=0.08, top=0.92, hspace=0.08)
fig.savefig(os.path.join(FIG_DIR, "fig4_cloud_phenology.png"), dpi=300,
            bbox_inches="tight", facecolor="white")
fig.savefig(os.path.join(FIG_DIR, "fig4_cloud_phenology.pdf"),
            bbox_inches="tight", facecolor="white")
plt.show()

# Clean up temp file
if os.path.exists(tmp_poly):
    os.remove(tmp_poly)
print("Figure 4 saved.")